In [20]:
import chromadb
import os
from dotenv import load_dotenv
from chromadb.utils import embedding_functions
from edgar import *
import yfinance as yf

Connect to the Chroma Vector DB

In [2]:
# Path to .env location containing the Chroma API keys
env_path = os.path.join("../data", ".env")

# Parse the .env file and retrieve the API keys
if load_dotenv(env_path):
    print("✅ Environment variables loaded from data/.env")
else:
    print("❌ Failed to load data/.env - Check if the file exists!")

CF_CLIENT_ID = os.getenv("CF-ACCESS-CLIENT-ID")
CF_CLIENT_SECRET = os.getenv("CF-ACCESS-CLIENT-SECRET")
CHROMA_URL = os.getenv("CHROMA_URL")

print(f"ID: {CF_CLIENT_ID}")
print(f"Secret: {CF_CLIENT_SECRET}")
print(f"Chroma URL: {CHROMA_URL}")


✅ Environment variables loaded from data/.env
ID: 15703ac0e5797dac885c851f20afe6cb.access
Secret: 9db6c201a502a20d0aa9eae4dac06d36b2f6a29b46201c20df2f5174263ddf57
Chroma URL: chroma.taskcomply.com


In [4]:
# 2. Setup the Client with Custom Headers
client = chromadb.HttpClient(
    host=CHROMA_URL,                              # Your Cloudflare URL
    port=443,                                     # Standard HTTPS port
    ssl=True,                                     # MUST be True for Cloudflare
    headers={
        "CF-Access-Client-Id": CF_CLIENT_ID,
        "CF-Access-Client-Secret": CF_CLIENT_SECRET
    },
)

# 3. Test the connection
print(f"Connection Heartbeat: {client.heartbeat()}")

Connection Heartbeat: 1775379133364967060


Determine embedding function and create 2 streams:
- Fundamentals
- Sentiment

In [6]:
# 4. Use a lightweight embedding function
# Default is 'all-MiniLM-L6-v2' which is great for financial headlines
emb_fn = embedding_functions.DefaultEmbeddingFunction()

# 3. Create the two streams
fundamentals_col = client.get_or_create_collection(
    name="stream1_fundamentals",
    embedding_function=emb_fn
)

sentiment_col = client.get_or_create_collection(
    name="stream2_sentiment",
    embedding_function=emb_fn
)

In [8]:
collections = client.list_collections()

print(collections)

[Collection(name=stream1_fundamentals), Collection(name=stream2_sentiment)]


1. Stream 1: Ingesting Fundamentals (The "Ground Truth")
We will use edgartools to pull the latest 10-Ks. For RAG to be effective, we don't just want the whole document; we want the Risk Factors (Item 1A), as these provide the best "analytical anchor."

In [14]:
# Set tickers to ingest
tickers = ["NVDA", "AAPL", "TSLA"]

# MUST set your identity for SEC compliance
set_identity("david.j.bain@student.uts.edu.au")

In [19]:
company = Company("NVDA")
# Get the latest Annual Report
filing = company.get_filings(form="10-K").latest()

# Extract the specific "Risk Factors" section as Markdown
# EdgarTools 2026 handles the section parsing automatically
tenk = filing.obj()

risk = tenk.risk_factors

print("Fundamentals Stream Hydrated.")

Fundamentals Stream Hydrated.


2. Stream 2: Ingesting Sentiment (The "Market Pulse")
For the sentiment stream, we want the most recent headlines. Since Tesla (TSLA) just had a delivery miss on April 2nd, and Apple (AAPL) just celebrated its 50th anniversary on April 1st, there is plenty of fresh data.

In [48]:
t = yf.Ticker('NVDA')
news = t.news # Returns a list of news dictionaries

docs = []
metas = []
ids = []

for i, item in enumerate(news):
    content = item.get("content") or {}
    provider = content.get("provider") or {}
    thumbnail = content.get("thumbnail") or {}

    docs.append(content.get("title"))

    metas.append({
        "ticker": ticker,
        "publisher": provider.get("displayName"),
        "url": thumbnail.get("originalUrl"),
        "stream": "sentiment"
    })

    ids.append(f"{ticker}_news_{i}_{item.get('id')}")

print("Sentiment Stream Hydrated.")

print(docs[0])
print(metas[0])
print(ids[0], "\n")

print(news[0])

Sentiment Stream Hydrated.
These 2 Monster Stocks Could Be the Best Investments You Make This Decade
{'ticker': 'TSLA', 'publisher': 'Motley Fool', 'url': 'https://media.zenfs.com/en/motleyfool.com/ba29f6baf5dc0cf4d930f6ae3fd2da13', 'stream': 'sentiment'}
TSLA_news_0_81ddd147-46dc-37e5-9309-5e2bf09458db 

{'id': '81ddd147-46dc-37e5-9309-5e2bf09458db', 'content': {'id': '81ddd147-46dc-37e5-9309-5e2bf09458db', 'contentType': 'STORY', 'title': 'These 2 Monster Stocks Could Be the Best Investments You Make This Decade', 'description': '', 'summary': 'As artificial intelligence (AI) infrastructure spending continues to accelerate, two members of the "Magnificent Seven" look particularly appealing as beneficiaries over the next decade.', 'pubDate': '2026-04-05T10:35:00Z', 'displayTime': '2026-04-05T10:35:00Z', 'isHosted': True, 'bypassModal': False, 'previewUrl': None, 'thumbnail': {'originalUrl': 'https://media.zenfs.com/en/motleyfool.com/ba29f6baf5dc0cf4d930f6ae3fd2da13', 'originalWidth': 